In [1]:
!pip install transformers

In [2]:
!pip install datasets

In [3]:
from datasets import load_dataset
import pandas as pd


dataset = load_dataset("stanfordnlp/imdb")


train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])


print(train_df.head())
print(test_df.head())

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


print(train_df.head(5))


print(test_df.head(5))

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0
                                                text  label
0  I love sci-fi and am willing to put up with a ...      0
1  Worth the entertainment value of a rental, esp...      0
2  its a totally average film with a few semi-alr...      0
3  STAR RATING: ***** Saturday Night **** Friday ...      0
4  First off let me say, If you haven't enjoyed a...      0
Train shape: (25000, 2)
Test shape: (25000, 2)
                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t... 

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
from datasets import Dataset

def tokenize_function(examples):
  return tokenizer(examples["text"], padding="max_length", truncation=True)


train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [8]:
tokenized_train

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})

In [21]:
tokenized_train[1]['input_ids'][0]

101

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/imdb_bert_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

In [30]:
training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start

In [31]:
from transformers import AutoModelForSequenceClassification,Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
)
print("model created ✅")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model created ✅


In [32]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)


In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.198287,0.178306


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1563, training_loss=0.2411741615485779, metrics={'train_runtime': 3553.4231, 'train_samples_per_second': 7.035, 'train_steps_per_second': 0.44, 'total_flos': 6577776384000000.0, 'train_loss': 0.2411741615485779, 'epoch': 1.0})

In [34]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
0.198287,0.178306,1


{'eval_loss': 0.17830581963062286}

In [35]:
save_path = "/content/drive/MyDrive/imdb_bert_final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/imdb_bert_final


In [36]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

save_path = "/content/drive/MyDrive/imdb_bert_final"

tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForSequenceClassification.from_pretrained(save_path)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("Model loaded on:", device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded on: cuda


In [37]:
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred = torch.argmax(probs).item()

    label = "POSITIVE 😊" if pred == 1 else "NEGATIVE 😞"
    return label, probs.cpu().numpy()

# ── Test it ─────────────────────────────────────────
review = "This movie was absolutely fantastic. The acting was superb!"
label, probs = predict(review)

print(f"Review: {review}")
print(f"Prediction: {label}")
print(f"Confidence: negative={probs[0]:.4f}, positive={probs[1]:.4f}")

Review: This movie was absolutely fantastic. The acting was superb!
Prediction: POSITIVE 😊
Confidence: negative=0.0043, positive=0.9957


In [38]:
reviews = [
    "This movie was absolutely fantastic. The acting was superb!",
    "Terrible film. I wasted two hours of my life.",
    "It was okay, not great but not bad either.",
    "A masterpiece. One of the best films I've ever seen.",
    "Boring and predictable. Do not recommend.",
]

for r in reviews:
    label, probs = predict(r)
    print(f"{label:15s} | {r[:60]}")

POSITIVE 😊      | This movie was absolutely fantastic. The acting was superb!
NEGATIVE 😞      | Terrible film. I wasted two hours of my life.
NEGATIVE 😞      | It was okay, not great but not bad either.
POSITIVE 😊      | A masterpiece. One of the best films I've ever seen.
NEGATIVE 😞      | Boring and predictable. Do not recommend.
